# Direct-CP benchmark toy fit for $B^\pm\to K^\pm\pi^+\pi^-$

The full paper-inspired amplitude composition is retained. `generate_cp_toy` and `CPFitSession` only simplify the workflow around the physics model. Toy generation uses the default inverse-transform sampler.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dalitzplotfitter import (
    BaBarFlatte, CPFitSession, CPRealImag, DecayChannel, DecayModel, LASS,
    NonResonant, Parameter, Resonance, enable_x64, generate_cp_toy, plot_dalitz,
)
enable_x64()


In [ ]:
truth_spec={
    "Kstar892":(1.00,0.00,+0.04,-0.03),
    "KpiS":(1.40,-0.60,-0.10,+0.08),
    "rho770":(0.65,0.10,+0.06,+0.04),
    "f0_980":(-0.20,1.00,-0.05,+0.07),
    "NR":(-0.50,0.10,0.00,0.00),
}
truth={}
shared={}
for name,(x,y,dx,dy) in truth_spec.items():
    pars=(
        Parameter.coefficient(f"{name}.x",x,owner=name,fixed=(name=="Kstar892"),step=0.01),
        Parameter.coefficient(f"{name}.y",y,owner=name,fixed=(name=="Kstar892"),step=0.01),
        Parameter.coefficient(f"{name}.dx",dx,owner=name,fixed=(name=="NR"),step=0.01),
        Parameter.coefficient(f"{name}.dy",dy,owner=name,fixed=(name=="NR"),step=0.01),
    )
    shared[name]=CPRealImag(*pars)
    truth.update({p.name:p.value for p in pars})

def components(q):
    c={name:coeff.for_charge(q) for name,coeff in shared.items()}
    return [
        Resonance("Kstar892",(0,2),c["Kstar892"],mass=0.8958,width=0.0474,spin=1,resonance_radius=4.0,parent_radius=4.0),
        Resonance("KpiS",(0,2),c["KpiS"],lineshape=LASS(2.07,3.32,1.8),mass=1.425,width=0.270,spin=0,resonance_radius=4.0,parent_radius=4.0),
        Resonance("rho770",(1,2),c["rho770"],mass=0.7753,width=0.1491,spin=1,resonance_radius=4.0,parent_radius=4.0),
        Resonance("f0_980",(1,2),c["f0_980"],lineshape=BaBarFlatte(),mass=0.965,width=0.0,spin=0,resonance_radius=4.0,parent_radius=4.0),
        NonResonant(c["NR"]),
    ]

plus_model=DecayModel(DecayChannel("B+",("K+","pi+","pi-")),components(+1),normalization_method="square-dalitz",normalization_resolution=350,normalization_pair=(0,2))
minus_model=DecayModel(DecayChannel("B-",("K-","pi-","pi+")),components(-1),normalization_method="square-dalitz",normalization_resolution=350,normalization_pair=(0,2))


In [ ]:
plus_data,minus_data=generate_cp_toy(plus_model,minus_model,40_000,parameters=truth,seed=505)
print("B+ / B-:",plus_data.size,minus_data.size)
fig,axes=plt.subplots(1,2,figsize=(12,5),constrained_layout=True)
plot_dalitz(plus_data,x="s13",y="s23",ax=axes[0],title="B+")
plot_dalitz(minus_data,x="s13",y="s23",ax=axes[1],title="B-")
plt.show()


In [ ]:
session=CPFitSession(plus_model,minus_model,plus_data,minus_data)
rng=np.random.default_rng(5051)
start={p.name:truth[p.name]+rng.normal(0,0.06) for p in session.parameters if not p.fixed}
result=session.fit(start,simplex=True,ncall=70_000)
session.report(result)
session.plot_projection(result,"s13")
plt.show()
session.plot_projection(result,"s23")
plt.show()
